#  Synthetic Datasat Generation

### Goal: Generate a Q&A dataset for GDPR

In [ ]:
!nvidia-smi -L

To get around annoying text-wrapping issues, run this cell.

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "your/data/path"

## Installing Dependencies

In [ ]:
!apt-get update
!apt-get install -y build-essential cmake git libcuda1-525 libcudnn8-dev ninja-build

In [ ]:
%env CC=/usr/bin/gcc
%env CXX=/usr/bin/g++

In [ ]:
nvidia_model = "A100" # @param ["A100", "T4"]
dcmake_cuda_ark = {"A100":80, "T4":75}[nvidia_model]

In [ ]:
!pip install --upgrade pip

## INSTALLING HUGGINGFACE
!pip install huggingface-hub

## INSTALLING CMAKE, NINJA, SCIKIT BUILD CORE
!pip install cmake ninja scikit-build-core json-repair pydantic

## INSTALLING llama-cpp-python
# GPU llama-cpp-python; Starting from version llama-cpp-python==0.1.79, it supports GGUF
!CMAKE_ARGS="-DGGML_CUDA=on  -DCMAKE_CUDA_ARCHITECTURES=80" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir > /dev/null

Next, you'll need to download the model weights from HuggingFace.

Here's a list of models you can choose from: https://huggingface.co/models?pipeline_tag=text-generation&sort=trending&search=GGUF

Note: The model you select **must** be of type "GGUF"

GGUF is...
- binary file format for storing models for inference
- designed for fast loading and saving of models
- easy to use (with a few lines of code)
- mmap (memory mapping) compatibility: models can be loaded using mmap for fast loading and saving.

In [ ]:
# @title Select Large Language Model
selected_llm = 'Llama-3.1-8B-Instruct' # @param ["Mistral-7B", "Llama-3.1-8B-Instruct", "Qwen3-8B-Q8", "Mistral-7B-OpenOrca", "Qwen3-8B", "DeepSeek-R1-Distill-Llama-8B-full-precision", "DeepSeek-R1-Distill-Llama-8B-quantized", "gpt-oss-20b"]

model_dic = {
    "Mistral-7B":{"HF_REPO_NAME":"TheBloke/Mistral-7B-Instruct-v0.1-GGUF","HF_MODEL_NAME":"mistral-7b-instruct-v0.1.Q4_K_M.gguf"},
    "Mistral-7B-OpenOrca":{"HF_REPO_NAME":"TheBloke/Mistral-7B-OpenOrca-GGUF","HF_MODEL_NAME":"mistral-7b-openorca.Q5_K_M.gguf"},
    "Qwen3-8B":{"HF_REPO_NAME":"Qwen/Qwen3-8B-GGUF","HF_MODEL_NAME":"Qwen3-8B-Q4_K_M.gguf"},
    "DeepSeek-R1-Distill-Llama-8B-full-precision": {"HF_REPO_NAME":"unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF","HF_MODEL_NAME":"DeepSeek-R1-Distill-Llama-8B-F16.gguf"},
    "DeepSeek-R1-Distill-Llama-8B-quantized": {"HF_REPO_NAME":"unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF","HF_MODEL_NAME":"DeepSeek-R1-Distill-Llama-8B-Q4_K_S.gguf"},
    "DeepSeek-R1-Distill-Llama-8B-quantized": {"HF_REPO_NAME":"unsloth/DeepSeek-R1-Distill-Llama-8B-GGUF","HF_MODEL_NAME":"DeepSeek-R1-Distill-Llama-8B-Q4_K_S.gguf"},
    "gpt-oss-20b":{"HF_REPO_NAME":"unsloth/gpt-oss-20b-GGUF", "HF_MODEL_NAME":"gpt-oss-20b-F16.gguf"},
    "Qwen3-8B-Q8": {"HF_REPO_NAME": "Qwen/Qwen3-8B-GGUF", "HF_MODEL_NAME": "Qwen3-8B-Q8_0.gguf"},
    "Llama-3.1-8B-Instruct": {"HF_REPO_NAME": "unsloth/Llama-3.1-8B-Instruct-GGUF", "HF_MODEL_NAME": "Llama-3.1-8B-Instruct-BF16.gguf"}
}


In [ ]:
import os

from huggingface_hub import hf_hub_download


HF_REPO_NAME = model_dic[selected_llm]['HF_REPO_NAME']
HF_MODEL_NAME = model_dic[selected_llm]['HF_MODEL_NAME']
LOCAL_DIR_NAME = "models"

os.makedirs(LOCAL_DIR_NAME, exist_ok=True)
model_path = hf_hub_download(
    repo_id=HF_REPO_NAME, filename=HF_MODEL_NAME, local_dir=LOCAL_DIR_NAME
)

Now, let's initialize the "Llama" framework.

So this is a bit messy.  Llama-cpp was named after Meta's open-source "*Llama*" LLMs.  The framework was built to make it easy to locally run & program with this LLM.  However, now, the framework as been abstracted and modified to work with ***any*** open-source text-generation LLM, as long as it is in the GGUF model file type.

In our case, we are using the Mistral open-source LLM and Llama-cpp as our framework.

In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_threads=2, # CPU cores
    n_batch=512, # Should be between 1 and n_ctx, consider the amount of VRAM in your GPU.
    n_gpu_layers=40, # The max for this model is 30 in a T4
    n_ctx=4096, # Context window
)


# Testing the LLM

In [ ]:
prompt = """
What is the US Department of Defense?
"""

In [ ]:
response = llm(prompt,stream=True,stop=["\n\n"],temperature=0.7, max_tokens=200)

In [ ]:
generated_text = ""
for output in response:
    result = output['choices'][0]['text']
    generated_text+=result
    print(result,end="")

# Q&A generation

## Dataset loader

In [ ]:
import os, json

# ==========================================
# DATA LOADING (THE GRAPH)
# ==========================================
def load_gdpr_graph(file_path):
    with open(file_path, 'r') as f:
        data = json.load(f)
    return data

## Prompts

In [ ]:
# ==========================================
# PROMPT TEMPLATES - full set, verbatim from thesis
# ==========================================

PROMPT_ARTICLE_UNITY_Q = """
You are a legal-question generator. Write one precise question about the GDPR
that, when answered correctly, must be the excerpt provided below from
Article {number} (or its faithful paraphrase).

Requirements:
- Base the question only on the inputs below; do not use external context.
- The question must uniquely target the "Required Answer" excerpt, not any
other part of the article.
- Avoid yes/no questions. Ask for a short factual answer (who/what/which/
under what conditions/according to which criteria).
- Keep the question under 50 words.
- Output only valid JSON as specified.

Article metadata (optional):
\"\"\"{metadata}\"\"\"
Article {number} (full text):
\"\"\"{full_text}\"\"\"
Contextual reference for framing the question (required excerpt):
\"\"\"{excerpt}\"\"\"

Output format (JSON):
{{
"article_number": {number},
"question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden

Validation checklist (do not include in output):
- Would the required excerpt alone fully answer the question?
- Is the question specific enough that a different annex passage would not fit?
- Is the question non-yes/no and <50 words?

"""

PROMPT_ARTICLE_UNITY_A = """
You are a legal assistant. Provide a precise and concise answer based
solely on the following excerpt from Article {number} of the GDPR.

Requirements:
- Base your answer only on the inputs below; do not use external context.
- The answer must exactly or faithfully paraphrase the "Required Answer"
excerpt.
- Keep the answer factual and concise.
- You must elaborate the answer as your own, do not copy it.

Article {number} (full text):
\"\"\"{full_text}\"\"\"
Required answer excerpt:
\"\"\"{excerpt}\"\"\"

Output format (JSON):
{{
"article_number": {number},
"answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden
"""

PROMPT_ARTICLE_RECITAL_BINDING_Q = """
You are a legal-question generator. Write one precise, legally relevant
question about the GDPR
that explicitly captures the relationship between the specified Recital and
the specified Article.

Requirements:
- The question must be answerable solely from Recital {recital_num} and Article {article_num} (below);
do not rely on external context.
- Make the linkage explicit (e.g., "According to Recital {recital_num}, how does it inform/clarify/apply
to Article {article_num} (text below) regarding X?").
- The question must require using both texts together (not just one of them).
- Avoid yes/no questions. Ask for a short factual answer (who/what/which/under what conditions/according to criteria).
- Keep the question under 50 words.
- Make it specific enough that a different recital or article passage would not fit.
- Output only valid JSON as specified.

Context:
Article metadata (optional):
\"\"\"{metadata}\"\"\"
Article {article_num} (full text):
\"\"\"{article_text}\"\"\"
Recital {recital_num} (full text):
\"\"\"{recital_text}\"\"\"

Output format (JSON):
{{
"recital_number": {recital_num},
"article_number": {article_num},
"question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden

Validation checklist (do not include in output):
- Would the required excerpt alone fully answer the question?
- Is the question specific enough that a different annex passage would not fit?
- Is the question non-yes/no and <50 words?
"""

PROMPT_ARTICLE_RECITAL_BINDING_A = """
You are a legal assistant. Provide a precise and concise answer based
solely on the following Recital {recital_number} and Article {article_number} of the GDPR.

Requirements:
- Base your answer only on the inputs below; do not use external context.
- The answer must explain faithfully how the recital informs, clarifies, or
applies to the article's provision.
- Keep the answer factual and concise.
- You must elaborate the answer as your own, do not copy it.

Article {article_number} (full text):
\"\"\"{article_text}\"\"\"

Recital {recital_number} (full text):
\"\"\"{recital_text}\"\"\"

Required answer excerpt:
\"\"\"{answer_excerpt}\"\"\"

Output format (JSON):
{{
  "recital_number": {recital_number},
  "article_number": {article_number},
  "answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

PROMPT_RECITAL_UNITY_Q = """
You are a legal-question generator. Task: write one precise question about
the GDPR whose correct answer is the provided recital (or its faithful paraphrase).

Requirements:
- Base the question only on the recital text given below; do not invent
facts outside it.
- The question must be answerable uniquely by that recital's content
and not by other recitals.
- Prefer specific "According to Recital {recital_number}..." phrasing.
- Avoid yes/no questions; ask for a short factual answer (who/what/when/where/which/under what conditions).
- Keep the question under 50 words.
- Output only valid JSON as specified.

Recital number: {recital_number}
Recital text:
\"\"\"{recital_text}\"\"\"

Output format (JSON):
{{
  "recital_number": {recital_number},
  "question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!

Validation checklist (do not include in output):
- Would the required excerpt alone fully answer the question?
- Is the question specific enough that a different annex passage would not fit?
- Is the question non-yes/no and <50 words?
"""

PROMPT_RECITAL_UNITY_A = """
You are a legal assistant. Provide a precise and concise answer based
solely on the following excerpt from Recital {recital_number} of the GDPR.

Requirements:
- Base your answer only on the excerpt; do not use external context.
- The answer must faithfully paraphrase or restate the recital's content.
- Keep the answer factual and concise.
- You must elaborate the answer as your own; do not copy it.

Recital {recital_number} (full text):
\"\"\"{recital_text}\"\"\"

Required answer excerpt:
\"\"\"{answer_excerpt}\"\"\"

Output format (JSON):
{{
  "recital_number": {recital_number},
  "answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

PROMPT_AUGMENTATION_Q = """
You are an assistant that augments questions about the GDPR.
Create variations of the given question that ask for the same information in different ways.

Rules:
- Do NOT directly reference any paragraph, article, annex, or recital.
- The answer's core meaning must remain the same.

Original question:
```
{original_question}
```
Full context:
```
{full_context}
```
Reference excerpt (original answer):
```
{original_answer}
```
Number of questions to generate: {n}

Generate alternative phrasings of the original question that ask for the same information but with different words.

Output format (JSON List):
{{
  "questions": [
    "<your augmented question 1>",
    "<your augmented question 2>",
    ...
    "<your augmented question n>"
  ]
}}
END_JSON

Return ONLY the questions in a JSON List format—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

PROMPT_ANNEX_RECITAL_BINDING_Q = """
You are a legal-question generator. Write one precise question about the EU AI Act that explicitly captures the relationship between the specified Recital and the specified Annex.

Requirements:
- The question must be answerable solely from the provided Recital {recital_num} and Annex {annex_num} texts; do not rely on external context.
- Make the linkage explicit (e.g., "According to Recital {recital_num}, how does it inform/clarify/apply to Annex {annex_num} (text below) regarding X?").
- The question must require using both texts together (not just one of them).
- Avoid yes/no questions. Ask for a short factual answer (who/what/which/under what conditions/according to which criteria).
- Keep the question under 50 words.
- Make it specific enough that a different recital or annex passage would not fit.
- Output only valid JSON as specified.

Annex {annex_num} (full text):
\"\"\"{annex_text}\"\"\"
Recital {recital_num} (full text):
\"\"\"{recital_text}\"\"\"

Output format (JSON):
{{
"recital_number": {recital_num},
"annex_number": {annex_num},
"question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!

Validation checklist (do not include in output):
- Does the question necessitate consulting both the recital and annex texts?
- Is the connection unique to these two inputs (not generic across the AI Act)?
- Is the question specific, non-yes/no, and <50 words?
"""

PROMPT_ANNEX_RECITAL_BINDING_A = """
You are a legal assistant. Provide a precise and concise answer based solely on the following Recital {recital_number} and Annex {annex_number} from the EU AI Act.

Requirements:
- Base your answer only on the provided texts; do not use external context.
- The answer must explicitly reflect the relationship between Recital {recital_number} and Annex {annex_number}.
- Keep the answer factual, short, and specific to these two inputs.
- Avoid generic statements or interpretations not grounded in the texts.
- You must elaborate the answer as your own, do not copy it.

Annex {annex_number} (full text):
\"\"\"{annex_text}\"\"\"
Recital {recital_number} (full text):
\"\"\"{recital_text}\"\"\"

Output format (JSON):
{{
"recital_number": {recital_number},
"annex_number": {annex_number},
"answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

PROMPT_ANNEX_ARTICLE_BINDING_Q = """
You are a legal-question generator. Write one precise question about the EU AI Act that explicitly captures the relationship between the specified Article and the specified Annex.

Requirements:
- The question must be answerable solely from the provided Article {article_num} and Annex {annex_num} texts; do not rely on external context.
- Make the linkage explicit (e.g., "According to Article {article_num}, how does it inform/clarify/apply to Annex {annex_num} (text below) regarding X?").
- The question must require using both texts together (not just one of them).
- Avoid yes/no questions. Ask for a short factual answer (who/what/which/under what conditions/according to which criteria).
- Keep the question under 50 words.
- Make it specific enough that a different article or annex passage would not fit.
- Output only valid JSON as specified.

Annex {annex_num} (full text):
\"\"\"{annex_text}\"\"\"
Article {article_num} (full text):
\"\"\"{article_text}\"\"\"

Output format (JSON):
{{
"article_number": {article_num},
"annex_number": {annex_num},
"question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!

Validation checklist (do not include in output):
- Does the question necessitate consulting both the article and annex texts?
- Is the connection unique to these two inputs (not generic across the AI Act)?
- Is the question specific, non-yes/no, and <50 words?
"""

PROMPT_ANNEX_ARTICLE_BINDING_A = """
You are a legal assistant. Provide a precise and concise answer based solely on the following Article {article_number} and Annex {annex_number} from the EU AI Act.

Requirements:
- Base your answer only on the provided texts; do not use external context.
- The answer must explicitly reflect the relationship between Article {article_number} and Annex {annex_number}.
- Keep the answer factual, short, and specific to these two inputs.
- Avoid generic statements or interpretations not grounded in the texts.
- You must elaborate the answer as your own, do not copy it.

Annex {annex_number} (full text):
\"\"\"{annex_text}\"\"\"
Article {article_number} (full text):
\"\"\"{article_text}\"\"\"

Output format (JSON):
{{
"article_number": {article_number},
"annex_number": {annex_number},
"answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

PROMPT_ANNEX_UNITY_Q = """
You are a legal-question generator. Write one precise question about the EU AI Act that, when answered correctly, must be the excerpt provided below from
Annex {number} (or its faithful paraphrase).

Requirements:
- Base the question only on the inputs below; do not use external context.
- The question must uniquely target the "Required Answer" excerpt, not any other part of the annex.
- Avoid yes/no questions. Ask for a short factual answer (who/what/which/under what conditions/according to which criteria).
- Keep the question under 50 words.
- Output only valid JSON as specified.

Annex {number} (full text):
\"\"\"{full_text}\"\"\"
Contextual reference for framing the question (required excerpt):
\"\"\"{excerpt}\"\"\"

Output format (JSON):
{{
"annex_number": {number},
"question": "<your question>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!

Validation checklist (do not include in output):
- Would the required excerpt alone fully answer the question?
- Is the question specific enough that a different annex passage would not fit?
- Is the question non-yes/no and <50 words?
"""

PROMPT_ANNEX_UNITY_A = """
You are a legal assistant. Provide a precise and concise answer based solely on the following excerpt from Annex {number} of the EU AI Act.

Requirements:
- Base your answer only on the provided text; do not use external context.
- The answer must faithfully paraphrase or restate the "Required Answer" excerpt.
- Keep the answer factual and concise.
- You must elaborate the answer as your own, do not copy it.

Annex {number} (full text):
\"\"\"{full_text}\"\"\"
Required answer excerpt:
\"\"\"{excerpt}\"\"\"

Output format (JSON):
{{
"annex_number": {number},
"answer": "<your concise factual answer>"
}}
END_JSON

!CRITICAL! Return ONLY the JSON—do NOT include any reasoning or extra text.
Any explanation is strictly forbidden!
"""

In [ ]:
import asyncio
from typing import List, Dict, Any, Callable, Type, Optional, Set
from pydantic import BaseModel, Field, ValidationError

# -----------------------------------------------------------------------------
# 1. Validation Schemas - Base Classes for DRY
# -----------------------------------------------------------------------------

class BaseQuestion(BaseModel):
    """Base class for all question schemas."""
    question: str

class BaseAnswer(BaseModel):
    """Base class for all answer schemas."""
    answer: str

# Unity schemas (single entity)
class ArticleUnityQuestion(BaseQuestion):
    article_number: int

class ArticleUnityAnswer(BaseAnswer):
    article_number: int

class RecitalUnityQuestion(BaseQuestion):
    recital_number: int

class RecitalUnityAnswer(BaseAnswer):
    recital_number: int

class AnnexUnityQuestion(BaseQuestion):
    annex_number: int

class AnnexUnityAnswer(BaseAnswer):
    annex_number: int

# Binding schemas (two entities)
class BindingArticleRecitalQuestion(BaseQuestion):
    recital_number: int
    article_number: int

class BindingArticleRecitalAnswer(BaseAnswer):
    recital_number: int
    article_number: int

class BindingAnnexArticleQuestion(BaseQuestion):
    annex_number: int
    article_number: int

class BindingAnnexArticleAnswer(BaseAnswer):
    annex_number: int
    article_number: int

class BindingAnnexRecitalQuestion(BaseQuestion):
    annex_number: int
    recital_number: int

class BindingAnnexRecitalAnswer(BaseAnswer):
    annex_number: int
    recital_number: int

class AugmentedVariants(BaseModel):
    questions: List[str] = Field(default_factory=list)

### Configuration Classes for Parameterization

In [ ]:
from dataclasses import dataclass

@dataclass
class UnityConfig:
    """Configuration for unity-type generation (article, recital, annex)."""
    data_key: str           # Key in self.data (e.g., 'articles', 'recitals')
    type_name: str          # Record type (e.g., 'article_unity')
    entity_name: str        # For logging (e.g., 'Article')
    text_key: str           # Key to extract text (e.g., 'fullText', 'text')
    sub_items_key: str      # Key for sub-items (e.g., 'paragraphs', 'sentences')
    sub_item_num_key: str   # Key for sub-item number in record
    question_schema: Type[BaseModel]
    answer_schema: Type[BaseModel]
    question_prompt_key: str
    answer_prompt_key: str
    q_format_fn: Callable   # Function to format question prompt
    a_format_fn: Callable   # Function to format answer prompt


@dataclass
class BindingConfig:
    """Configuration for binding-type generation (article-recital, annex-article, etc.)."""
    primary_data_key: str   # Key for primary entity (e.g., 'articles')
    secondary_data_key: str # Key for secondary entity lookup
    relation_key: str       # Key in primary for related items (e.g., 'relatedRecitals')
    type_name: str          # Record type
    primary_name: str       # Name for logging/record
    secondary_name: str     # Name for logging/record
    primary_text_key: str   # Key to extract primary text
    secondary_text_key: str # Key to extract secondary text
    question_schema: Type[BaseModel]
    answer_schema: Type[BaseModel]
    question_prompt_key: str
    answer_prompt_key: str
    q_format_fn: Callable   # Function to format question prompt
    a_format_fn: Callable   # Function to format answer prompt

### Generation Pipeline

In [ ]:
class AsyncQAGDPRPipeline:
    def __init__(
        self,
        data: Dict[str, Any],
        llm_func: Callable,
        output_path: str = "dataset.jsonl",
        max_concurrency: int = 1,
        prompts: Dict[str, str] = None
    ):
        self.data = data
        self.dataset: List[Dict[str, Any]] = []
        self.llm_func = llm_func
        self.output_path = output_path
        self.sem = asyncio.Semaphore(max_concurrency)
        self.write_lock = asyncio.Lock()
        self.prompts = prompts or {}
        self.processed_ids: Set[str] = self._load_progress()
        print(f"[INIT] Pipeline initialized. Resuming with {len(self.processed_ids)} items.")

    # --- Persistence ---
    def _load_progress(self) -> Set[str]:
        ids = set()
        if not os.path.exists(self.output_path):
            return ids
        try:
            with open(self.output_path, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        self.dataset.append(record)
                        unique_key = f"{record.get('type')}|{record.get('ref_id')}"
                        ids.add(unique_key)
                    except json.JSONDecodeError:
                        continue
        except Exception as e:
            print(f"[WARN] Error reading file: {e}")
        return ids

    async def _save_record(self, record: Dict[str, Any]):
        unique_key = f"{record.get('type')}|{record.get('ref_id')}"
        if unique_key in self.processed_ids:
            return

        try:
            self.dataset.append(record)
            async with self.write_lock:
                with open(self.output_path, 'a', encoding='utf-8') as f:
                    f.write(json.dumps(record) + "\n")
                self.processed_ids.add(unique_key)
        except Exception as e:
            print(f"[ERROR] Save failed: {e}")

    # --- Core LLM ---
    async def _generate_structured(
        self,
        prompt: str,
        schema: Type[BaseModel],
        retries: int = 2
    ) -> Optional[BaseModel]:
        async with self.sem:
            for attempt in range(retries + 1):
                try:
                    text_response = await self.llm_func(
                        prompt=prompt,
                        temperature=0.7,
                        max_tokens=512,
                        stop=["END_JSON"]
                    )

                    start = text_response.find("{")
                    if start != -1:
                        json_str = text_response[start:]
                        if not json_str.strip().endswith("}"):
                            json_str += "}"
                    else:
                        json_str = text_response

                    try:
                        data = json.loads(json_str)
                    except json.JSONDecodeError:
                        data = json.loads(repair_json(json_str))

                    return schema(**data)

                except (ValidationError, Exception) as e:
                    if attempt < retries:
                        continue
                    print(f"[FAIL] {e}")
                    return None
        return None

    # --- Generic Unity Generation ---
    async def _generate_unity(self, config: UnityConfig, limit: int = None, n: int = 5):
        """Generic method to generate unity-type Q&A for articles, recitals, or annexes."""
        print(f"--- {config.entity_name} Unity ---")

        items = self.data.get(config.data_key, [])
        if limit:
            items = items[:limit]

        tasks = []
        for item in items:
            item_num = item['number']
            if f"{config.type_name}|{item_num}" in self.processed_ids:
                continue

            # Process main item
            tasks.append(self._process_unity_item(config, item))

            # Process sub-items (paragraphs/sentences)
            sub_items = item.get(config.sub_items_key, [])
            for idx, sub_item in enumerate(sub_items):
                sub_text = sub_item.get('text', sub_item) if isinstance(sub_item, dict) else sub_item
                tasks.append(self._process_unity_item(config, {
                    'number': item_num,
                    config.text_key: sub_text,
                    config.sub_item_num_key: idx
                }))

        assert n > 0
        if tasks:
            await asyncio.gather(*(tasks * n))

    async def _process_unity_item(self, config: UnityConfig, item: Dict[str, Any]):
        """Process a single unity item (article/recital/annex or their sub-parts)."""
        item_num = item['number']
        sub_num = item.get(config.sub_item_num_key, 0)
        text = item.get(config.text_key, '')

        # Generate Question
        q_prompt = config.q_format_fn(self.prompts[config.question_prompt_key], item_num, text)
        q_res = await self._generate_structured(q_prompt, config.question_schema)
        if not q_res:
            return

        # Generate Answer
        a_prompt = config.a_format_fn(self.prompts[config.answer_prompt_key], item_num, text)
        a_res = await self._generate_structured(a_prompt, config.answer_schema)
        if not a_res:
            return

        ref_id = f"{item_num}_{sub_num}" if sub_num else item_num
        await self._save_record({
            "type": config.type_name,
            "ref_id": ref_id,
            "question": q_res.question,
            "answer": a_res.answer
        })
        print(f"[OK] {config.entity_name} {item_num}")

    # --- Generic Binding Generation ---
    async def _generate_binding(self, config: BindingConfig, limit: int = None, n: int = 5):
        """Generic method to generate binding Q&A between two entity types."""
        print(f"--- Binding {config.primary_name} - {config.secondary_name} Questions ---")

        # Build lookup for secondary entity
        secondary_items = self.data.get(config.secondary_data_key, [])
        secondary_lookup = {
            item['number']: item.get(config.secondary_text_key, '')
            for item in secondary_items
        }

        primary_items = self.data.get(config.primary_data_key, [])
        if limit:
            primary_items = primary_items[:limit]

        tasks = []
        for primary in primary_items:
            primary_num = primary['number']
            primary_text = primary.get(config.primary_text_key, '')

            for secondary_num in primary.get(config.relation_key, []):
                unique_key = f"{config.type_name}|{primary_num}_{secondary_num}"
                if unique_key in self.processed_ids:
                    continue

                secondary_text = secondary_lookup.get(secondary_num)
                if not secondary_text:
                    continue

                tasks.append(self._process_binding_item(
                    config, primary_num, primary_text, secondary_num, secondary_text
                ))

        assert n > 0
        if tasks:
            await asyncio.gather(*(tasks * n))

    async def _process_binding_item(
        self,
        config: BindingConfig,
        primary_num: int,
        primary_text: str,
        secondary_num: int,
        secondary_text: str
    ):
        """Process a single binding item."""
        # Generate Question
        q_prompt = config.q_format_fn(
            self.prompts[config.question_prompt_key],
            primary_num, primary_text, secondary_num, secondary_text
        )
        q_res = await self._generate_structured(q_prompt, config.question_schema)
        if not q_res:
            return

        # Generate Answer
        a_prompt = config.a_format_fn(
            self.prompts[config.answer_prompt_key],
            primary_num, primary_text, secondary_num, secondary_text
        )
        a_res = await self._generate_structured(a_prompt, config.answer_schema)
        if not a_res:
            return

        await self._save_record({
            "type": config.type_name,
            "ref_id": f"{primary_num}_{secondary_num}",
            config.primary_name.lower(): primary_num,
            config.secondary_name.lower(): secondary_num,
            "question": q_res.question,
            "answer": a_res.answer
        })
        print(f"[OK] Binding {primary_num}-{secondary_num}")

    # --- Public API Methods ---
    async def generate_article_unity(self, limit: int = None, n: int = 5):
        config = UnityConfig(
            data_key='articles',
            type_name='article_unity',
            entity_name='Article',
            text_key='fullText',
            sub_items_key='paragraphs',
            sub_item_num_key='paragraph_num',
            question_schema=ArticleUnityQuestion,
            answer_schema=ArticleUnityAnswer,
            question_prompt_key='PROMPT_ARTICLE_UNITY_Q',
            answer_prompt_key='PROMPT_ARTICLE_UNITY_A',
            q_format_fn=lambda p, num, txt: p.format(number=num, metadata="", full_text=txt, excerpt=txt),
            a_format_fn=lambda p, num, txt: p.format(number=num, full_text=txt, excerpt=txt),
        )
        await self._generate_unity(config, limit, n)

    async def generate_recital_unity(self, limit: int = None, n: int = 5):
        config = UnityConfig(
            data_key='recitals',
            type_name='recital_unity',
            entity_name='Recital',
            text_key='text',
            sub_items_key='sentences',
            sub_item_num_key='sentence_num',
            question_schema=RecitalUnityQuestion,
            answer_schema=RecitalUnityAnswer,
            question_prompt_key='PROMPT_RECITAL_UNITY_Q',
            answer_prompt_key='PROMPT_RECITAL_UNITY_A',
            q_format_fn=lambda p, num, txt: p.format(recital_number=num, recital_text=txt),
            a_format_fn=lambda p, num, txt: p.format(recital_number=num, recital_text=txt, answer_excerpt=txt),
        )
        await self._generate_unity(config, limit, n)

    async def generate_annex_unity(self, limit: int = None, n: int = 5):
        config = UnityConfig(
            data_key='annexes',
            type_name='annex_unity',
            entity_name='Annex',
            text_key='fullText',
            sub_items_key='paragraphs',
            sub_item_num_key='paragraph_num',
            question_schema=AnnexUnityQuestion,
            answer_schema=AnnexUnityAnswer,
            question_prompt_key='PROMPT_ANNEX_UNITY_Q',
            answer_prompt_key='PROMPT_ANNEX_UNITY_A',
            q_format_fn=lambda p, num, txt: p.format(number=num, metadata="", full_text=txt, excerpt=txt),
            a_format_fn=lambda p, num, txt: p.format(number=num, full_text=txt, excerpt=txt),
        )
        await self._generate_unity(config, limit, n)

    async def generate_binding_article_recital_questions(self, limit: int = None, n: int = 5):
        config = BindingConfig(
            primary_data_key='articles',
            secondary_data_key='recitals',
            relation_key='relatedRecitals',
            type_name='binding_question_article_recital',
            primary_name='Article',
            secondary_name='Recital',
            primary_text_key='fullText',
            secondary_text_key='text',
            question_schema=BindingArticleRecitalQuestion,
            answer_schema=BindingArticleRecitalAnswer,
            question_prompt_key='PROMPT_ARTICLE_RECITAL_BINDING_Q',
            answer_prompt_key='PROMPT_ARTICLE_RECITAL_BINDING_A',
            q_format_fn=lambda p, art, art_txt, rec, rec_txt: p.format(
                recital_num=rec, article_num=art, metadata="",
                article_text=art_txt, recital_text=rec_txt
            ),
            a_format_fn=lambda p, art, art_txt, rec, rec_txt: p.format(
                recital_number=rec, article_number=art,
                article_text=art_txt, recital_text=rec_txt, answer_excerpt=rec_txt
            ),
        )
        await self._generate_binding(config, limit, n)

    async def generate_binding_annex_article_questions(self, limit: int = None, n: int = 5):
        config = BindingConfig(
            primary_data_key='annexes',
            secondary_data_key='articles',
            relation_key='articles',
            type_name='binding_question_annex_article',
            primary_name='Annex',
            secondary_name='Article',
            primary_text_key='fullText',
            secondary_text_key='fullText',
            question_schema=BindingAnnexArticleQuestion,
            answer_schema=BindingAnnexArticleAnswer,
            question_prompt_key='PROMPT_ANNEX_ARTICLE_BINDING_Q',
            answer_prompt_key='PROMPT_ANNEX_ARTICLE_BINDING_A',
            q_format_fn=lambda p, ann, ann_txt, art, art_txt: p.format(
                annex_num=ann, article_num=art, metadata="",
                article_text=art_txt, annex_text=ann_txt
            ),
            a_format_fn=lambda p, ann, ann_txt, art, art_txt: p.format(
                annex_number=ann, article_number=art,
                article_text=art_txt, annex_text=ann_txt, answer_excerpt=ann_txt
            ),
        )
        await self._generate_binding(config, limit, n)

    async def generate_binding_annex_recital_questions(self, limit: int = None, n: int = 5):
        config = BindingConfig(
            primary_data_key='annexes',
            secondary_data_key='recitals',
            relation_key='recitals',
            type_name='binding_question_annex_recital',
            primary_name='Annex',
            secondary_name='Recital',
            primary_text_key='fullText',
            secondary_text_key='text',
            question_schema=BindingAnnexRecitalQuestion,
            answer_schema=BindingAnnexRecitalAnswer,
            question_prompt_key='PROMPT_ANNEX_RECITAL_BINDING_Q',
            answer_prompt_key='PROMPT_ANNEX_RECITAL_BINDING_A',
            q_format_fn=lambda p, ann, ann_txt, rec, rec_txt: p.format(
                annex_num=ann, recital_num=rec, metadata="",
                recital_text=rec_txt, annex_text=ann_txt
            ),
            a_format_fn=lambda p, ann, ann_txt, rec, rec_txt: p.format(
                annex_number=ann, recital_number=rec,
                recital_text=rec_txt, annex_text=ann_txt, answer_excerpt=rec_txt
            ),
        )
        await self._generate_binding(config, limit, n)

    # --- Augmentation ---
    async def augment_dataset(self, n: int = 5):
        """Augment existing Q&A pairs with question variations."""
        print("--- Augmentation ---")
        if not os.path.exists(self.output_path):
            return

        with open(self.output_path, 'r') as f:
            lines = f.readlines()

        tasks = []
        for line in lines:
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            if "augmented" in item['type']:
                continue

            if f"{item['type']}_augmented|{item['ref_id']}_aug_0" in self.processed_ids:
                continue

            tasks.append(self._process_augmentation(item, n))

        if tasks:
            await asyncio.gather(*tasks)

    async def _process_augmentation(self, item: Dict[str, Any], n: int = 5):
        prompt = self.prompts['PROMPT_AUGMENTATION_Q'].format(
            original_question=item['question'],
            full_context="",
            original_answer=item['answer'],
            n=n
        )
        variants = await self._generate_structured(prompt, AugmentedVariants)

        if variants:
            for i, q_var in enumerate(variants.questions):
                await self._save_record({
                    "type": f"{item['type']}_augmented",
                    "ref_id": f"{item['ref_id']}_aug_{i}",
                    "question": q_var,
                    "answer": item['answer'],
                    "original_ref_type": item['type']
                })
            print(f"[OK] Augmented {item['ref_id']}")


### Define the async wrapper function

In [ ]:
from functools import partial

async def llama_cpp_func(prompt, stop=None, **kwargs):
    """
    Wraps the blocking llm() call to run in a separate thread,
    making it compatible with asyncio.
    """
    loop = asyncio.get_running_loop()

    # Partial creates a callable with your specific arguments
    # strict=False is often safer if kwargs contains extra params
    call_func = partial(
        llm,
        prompt=prompt,
        stop=stop or [],
        **kwargs
    )

    # run_in_executor(None, ...) uses the default ThreadPoolExecutor
    response = await loop.run_in_executor(None, call_func)

    # Extract text from standard OpenAI-compatible response format
    return response['choices'][0]['text']


### Main

In [ ]:
import json
import os
import asyncio
from typing import Dict, Any, List, Type, Optional, Callable, Set
from json_repair import repair_json
from pydantic import BaseModel, ValidationError

gdpr_data = load_gdpr_graph('GDPR/datasets/gdpr_w_annexes.json')

# Configuration
OUTPUT_PATH = "GDPR/datasets/full_dataset.jsonl"
MAX_CONCURRENCY = 1  # Adjust based on your API rate limits
LIMIT = None  # Set to a number to limit items per category (useful for testing)
N_REPETITIONS = 1  # Number of times to generate Q&A for each item
N_AUGMENTATIONS = 3  # Number of augmented variants per question

# Prepare prompts dictionary
prompts_dict = {
  'PROMPT_ARTICLE_UNITY_Q': PROMPT_ARTICLE_UNITY_Q,
  'PROMPT_ARTICLE_UNITY_A': PROMPT_ARTICLE_UNITY_A,
  'PROMPT_RECITAL_UNITY_Q': PROMPT_RECITAL_UNITY_Q,
  'PROMPT_RECITAL_UNITY_A': PROMPT_RECITAL_UNITY_A,
  'PROMPT_ANNEX_UNITY_Q': PROMPT_ANNEX_UNITY_Q,
  'PROMPT_ANNEX_UNITY_A': PROMPT_ANNEX_UNITY_A,
  'PROMPT_ARTICLE_RECITAL_BINDING_Q': PROMPT_ARTICLE_RECITAL_BINDING_Q,
  'PROMPT_ARTICLE_RECITAL_BINDING_A': PROMPT_ARTICLE_RECITAL_BINDING_A,
  'PROMPT_ANNEX_ARTICLE_BINDING_Q': PROMPT_ANNEX_ARTICLE_BINDING_Q,
  'PROMPT_ANNEX_ARTICLE_BINDING_A': PROMPT_ANNEX_ARTICLE_BINDING_A,
  'PROMPT_ANNEX_RECITAL_BINDING_Q': PROMPT_ANNEX_RECITAL_BINDING_Q,
  'PROMPT_ANNEX_RECITAL_BINDING_A': PROMPT_ANNEX_RECITAL_BINDING_A,
  'PROMPT_AUGMENTATION_Q': PROMPT_AUGMENTATION_Q,
}

# Initialize pipeline
print(f"Initializing pipeline (output: {OUTPUT_PATH})...")
pipeline = AsyncQAGDPRPipeline(
  data=gdpr_data,
  llm_func=llama_cpp_func,
  output_path=OUTPUT_PATH,
  max_concurrency=MAX_CONCURRENCY,
  prompts=prompts_dict
)

print(len(pipeline.dataset))
print(pipeline.dataset[:3])

In [ ]:
# Run all generation steps
try:
    # Step 1: Generate Article Unity Q&A
    print("\n" + "="*60)
    print("STEP 1: Generating Article Unity Q&A")
    print("="*60)
    await pipeline.generate_article_unity(limit=LIMIT, n=N_REPETITIONS)

    # Step 2: Generate Recital Unity Q&A
    print("\n" + "="*60)
    print("STEP 2: Generating Recital Unity Q&A")
    print("="*60)
    await pipeline.generate_recital_unity(limit=LIMIT, n=N_REPETITIONS)

    # Step 3: Generate Annex Unity Q&A
    print("\n" + "="*60)
    print("STEP 3: Generating Annex Unity Q&A")
    print("="*60)
    await pipeline.generate_annex_unity(limit=LIMIT, n=N_REPETITIONS)

    # Step 4: Generate Binding Article-Recital Q&A
    print("\n" + "="*60)
    print("STEP 4: Generating Binding Article-Recital Q&A")
    print("="*60)
    await pipeline.generate_binding_article_recital_questions(limit=LIMIT, n=N_REPETITIONS)

    # Step 5: Generate Binding Annex-Article Q&A
    print("\n" + "="*60)
    print("STEP 5: Generating Binding Annex-Article Q&A")
    print("="*60)
    await pipeline.generate_binding_annex_article_questions(limit=LIMIT, n=N_REPETITIONS)

    # Step 6: Generate Binding Annex-Recital Q&A
    print("\n" + "="*60)
    print("STEP 6: Generating Binding Annex-Recital Q&A")
    print("="*60)
    await pipeline.generate_binding_annex_recital_questions(limit=LIMIT, n=N_REPETITIONS)

    # Step 7: Augment all generated Q&A
    print("\n" + "="*60)
    print("STEP 7: Augmenting Dataset")
    print("="*60)
    await pipeline.augment_dataset(n=N_AUGMENTATIONS)

    # Summary
    print("\n" + "="*60)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("="*60)
    print(f"Total records generated: {len(pipeline.dataset)}")
    print(f"Output saved to: {OUTPUT_PATH}")

except Exception as e:
    print(f"\n[ERROR] Pipeline failed: {e}")
    raise

In [ ]:
count_item = {"total": len(pipeline.dataset)}
for item in pipeline.dataset:
  if count_item.get(item['type']):
    count_item[item['type']] += 1
  else:
    count_item[item['type']] = 1
print(count_item)

In [ ]:
from itertools import filterfalse

def delete_category(category_name: str):
  pipeline.dataset = list(filterfalse(lambda x: x['type'] == category_name, pipeline.dataset))

## Truncate dataset

In [ ]:
import json
from itertools import filterfalse

DATASET_PATH = "GDPR/datasets/full_dataset.jsonl"
OUTPUT_PATH = "GDPR/datasets/truncated_dataset.jsonl"

dataset = []
with open(DATASET_PATH, 'r') as f:
  for line in f:
    try:
      record = json.loads(line)
      dataset.append(record)
    except json.JSONDecodeError:
      raise Exception("Cannot decode full dataset")


dataset = list(filterfalse(lambda x: "augmented" not in x["type"], dataset))

with open(OUTPUT_PATH, "a+", encoding='utf-8') as f:
  try:
    for record in dataset:
      f.write(json.dumps(record) + "\n")
  except Exception as e:
            print(f"[ERROR] Save failed: {e}")
